<a href="https://colab.research.google.com/github/sit-sophiahsg/3001-CVDL/blob/main/AAI3001_P1_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **P1-9 AAI3001 Computer Vision & Deep Learning Project**

| Student ID | Member |
|---| --- |
|  | Gador Sophia Helen Santiago |
| | Natalie Peh |
|  |Tan Jian Ying Ivan |
|  | Tan Jing Yu |
|| Tan Jun Ying  |

## **Market Produce Quality Detection & Dynamic Pricing**

**Background**




**Goals**

# **Setup & Reproducibility**

This section installs the packages, fixes every path, class list and seed in one `CFG`, and writes the helper files that every later section uses. This section is only ran once at the start. Later sections need only their own restore cell.

## Setup

In [1]:
# Cell 1.1 · Install packages

import importlib.metadata as md, subprocess, sys

def ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

tf_before = ver('transformers')
pkgs = ['ultralytics', 'transformers>=4.56', 'segmentation-models-pytorch', 'onnx', 'onnxruntime',
        'onnxscript', 'kagglehub', 'grad-cam', 'huggingface_hub', 'pyyaml']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

for p in ['torch', 'torchvision', 'ultralytics', 'transformers', 'segmentation-models-pytorch',
          'onnx', 'onnxruntime', 'kagglehub', 'grad-cam']:
    print(f'{p:28s} {ver(p)}')
if 'transformers' in sys.modules and ver('transformers') != tf_before:
    print('\n⚠️ transformers was upgraded after it was imported: Runtime ▸ Restart session, then run from cell 1.2.')

torch                        2.14.1+cpu
torchvision                  0.29.1+cpu
ultralytics                  8.4.216
transformers                 5.22.1
segmentation-models-pytorch  0.6.0
onnx                         1.23.2
onnxruntime                  1.32.0
kagglehub                    1.0.3
grad-cam                     1.5.7


In [2]:
# Cell 1.2 · Imports and device check

import os, sys, json, time, math, random, copy, shutil, zipfile, glob
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'none'
print(f'device: {device} | GPU: {gpu} | torch {torch.__version__} | torchvision {torchvision.__version__}')
if device == 'cpu':
    print('⚠️ No GPU. Runtime ▸ Change runtime type ▸ T4 GPU before training (not needed with REBUILD = False).')

device: cpu | GPU: none | torch 2.14.1+cpu | torchvision 0.29.1+cpu
⚠️ No GPU. Runtime ▸ Change runtime type ▸ T4 GPU before training (not needed with REBUILD = False).


The notebook only uses one configuration file, set up below. Every path, the class order, the tier thresholds and the seed are set once here and saved to `repo/configs/cfg.json`. The restore cells and the app read that file, so nothing is typed twice.

In [3]:
# Cell 1.3 · CFG: every path, class list, tier rule and seed in one place

ROOT = Path('/content/drive/MyDrive/AAI3001_Group9')
REBUILD = False

rel = dict(data='data', raw_timelapse='data/raw/timelapse', raw_pile='data/raw/pile', raw_bag='data/raw/bag',
           labels='data/labels', frames='data/frames', models='models', deploy='deploy',
           figures='figures', repo='repo', results='results.json',
           hurdle_manifest='data/split_manifest.csv', hurdle_zip='data/produce_jpg.zip',
           hurdle_ckpt='models/best_model_resnet50.pt')

CFG = dict(
    DEBUG=False,      # True = tiny subsets for testing code, logs to results_debug.json
    debug_n=32,       # items kept per class (or per video) when DEBUG is True
    seed=42,
    root=str(ROOT),
    paths={k: str(ROOT / v) for k, v in rel.items()}
          | dict(work='/content/work', kaggle_local='/content/produce_jpg'),
    produce_classes=['apple', 'banana', 'orange', 'capsicum', 'tomato', 'potato'],
    hurdle_produce_classes=['apple', 'banana', 'bitter_gourd', 'capsicum', 'cucumber',
                            'okra', 'orange', 'potato', 'tomato'],      # order the hurdle model was trained with
    hurdle_stage_classes=['Fresh', 'Rotten'],
    tiers=['A', 'B', 'C', 'D'],
    tier_min_freshness=dict(A=0.70, B=0.40, C=0.15, D=0.0),   # first tier whose minimum the freshness reaches
    tier_discount=dict(A=0.0, B=0.20, C=0.50, D=None),        # D = not for sale
    img=dict(size=224, resize=255, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    timelapse=dict(frames_per_day=4, forecast_gaps=[1, 3, 7], videos_per_type=3),
    gen_size=128,
    batch_size=32, num_workers=2,
)
PRODUCE, TIERS = CFG['produce_classes'], CFG['tiers']
print(f"REBUILD={REBUILD} | DEBUG={CFG['DEBUG']} | seed={CFG['seed']} | {len(PRODUCE)} produce types: {PRODUCE}")
print('tiers:', {t: (CFG['tier_min_freshness'][t], CFG['tier_discount'][t]) for t in TIERS})

REBUILD=False | DEBUG=False | seed=42 | 6 produce types: ['apple', 'banana', 'orange', 'capsicum', 'tomato', 'potato']
tiers: {'A': (0.7, 0.0), 'B': (0.4, 0.2), 'C': (0.15, 0.5), 'D': (0.0, None)}


In [4]:
# Cell 1.4 · Mount Drive, create the folder layout, check the hurdle files we reuse

from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
assert ROOT.is_dir(), f'{ROOT} not found: check the folder name in cell 1.3, or add a shortcut to My Drive'

P = CFG['paths']
for k in ['data', 'raw_timelapse', 'raw_pile', 'raw_bag', 'labels', 'frames', 'models', 'deploy', 'figures', 'work']:
    Path(P[k]).mkdir(parents=True, exist_ok=True)
for sub in ['src', 'configs', 'app', 'notebooks']:
    (ROOT / 'repo' / sub).mkdir(parents=True, exist_ok=True)
(ROOT / 'repo' / 'src' / '__init__.py').touch()

def size_mb(p):
    return round(Path(p).stat().st_size / 1e6, 1) if Path(p).exists() else None

rows = [(k, P[k], size_mb(P[k])) for k in ['hurdle_manifest', 'hurdle_zip', 'hurdle_ckpt']]
print(pd.DataFrame(rows, columns=['file', 'path', 'MB']).to_string(index=False), '\n')

if Path(P['hurdle_manifest']).exists():
    m = pd.read_csv(P['hurdle_manifest'])
    print(f"manifest: {len(m)} rows | splits {m['split'].value_counts().to_dict()}")
    print('has jpg_rel column:', 'jpg_rel' in m.columns)
if Path(P['hurdle_ckpt']).exists():
    sd = torch.load(P['hurdle_ckpt'], map_location='cpu')
    heads = {k: tuple(v.shape) for k, v in sd.items() if 'head' in k and k.endswith('weight')}
    print(f'checkpoint: {len(sd)} tensors | head weights: {heads}')

videos = sorted(p.name for p in Path(P['raw_timelapse']).glob('*') if p.is_file())
print(f"\nraw/timelapse: {len(videos)} files | videos.csv present: {(Path(P['labels']) / 'videos.csv').exists()}")
print(f"raw/pile: {sum(1 for _ in Path(P['raw_pile']).rglob('*.*'))} files | raw/bag: {sum(1 for _ in Path(P['raw_bag']).rglob('*.*'))} files")
missing = [k for k, _, mb in rows if mb is None]
print('\n✅ all hurdle files found' if not missing else f'\n⚠️ missing: {missing}')

Mounted at /content/drive
           file                                                                path    MB
hurdle_manifest       /content/drive/MyDrive/AAI3001_Group9/data/split_manifest.csv   4.9
     hurdle_zip          /content/drive/MyDrive/AAI3001_Group9/data/produce_jpg.zip 863.4
    hurdle_ckpt /content/drive/MyDrive/AAI3001_Group9/models/best_model_resnet50.pt  94.5 

manifest: 30357 rows | splits {'train': 21289, 'val': 4557, 'test': 4511}
has jpg_rel column: True
checkpoint: 340 tensors | head weights: {'produce_head.weight': (9, 2048), 'stage_head.weight': (2, 2048)}

raw/timelapse: 18 files | videos.csv present: True
raw/pile: 0 files | raw/bag: 0 files

✅ all hurdle files found


## Helper Files

Code used by more than one section lives in small `.py` files, each written by exactly one cell of this notebook:

| File | Written in | Used for |
|---|---|---|
| `repo/notebooks/nbtools.py` | 1.5 | seeding, `log_result`, `save_fig`, saved-output checks (`build_needed`, `may_write`, `show_saved`) |
| `repo/notebooks/restore.py` | 1.6 | the body of every restore cell |
| `repo/src/preprocess.py` | 1.8 | the one image transform shared by the notebook and the app |
| `repo/notebooks/labeltools.py` | 2.7 | Grounding DINO + SAM 2 tracking |
| `repo/notebooks/cliptools.py` | 2.8 | CLIP zero-shot fresh/rotten |
| `repo/src/clean.py` | 2.11 | black-border removal |
| `repo/notebooks/freeze.py` | 2.12 | test-set fingerprints |
| `repo/notebooks/freshcache.py` | 2.13 | freshness training cache |
| `repo/src/augment.py` | 3.2 | bag overlay, training transform |
| `repo/notebooks/dettools.py` | 4 | detector scoring |
| `freshdata.py`, `freshmodel.py`, `freshtrain.py` | 5 | freshness data, model, training loop |
| `repo/notebooks/segtools.py` | 6 | U-Net data, losses, scores |
| `repo/notebooks/gentools.py` | 7 | pix2pix generator and discriminator |
| `repo/src/pricing.py` | 8 | the scale rules (also used by the app) |

In [5]:
%%writefile /content/drive/MyDrive/AAI3001_Group9/repo/notebooks/nbtools.py

# Cell 1.5 · nbtools.py

"""Notebook helpers: seeding, config save/load, results log, figure saving, saved-output checks.
Used by the notebook only (the app never imports this file)."""
import json, os, random
from datetime import datetime
from pathlib import Path
import numpy as np

CFG = None

def init(cfg):
    """Give the helpers the notebook's CFG (call once per runtime; the restore cell does it)."""
    global CFG
    CFG = cfg

def rebuild():
    return bool(CFG.get('REBUILD', False))

def seed_everything(seed):
    """Seed every random generator for reproducible runs (as in the hurdle and Labs 2-3)."""
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

def save_cfg(cfg, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps({k: v for k, v in cfg.items() if k != 'REBUILD'}, indent=2))

def load_cfg(path):
    return json.loads(Path(path).read_text())

def _sec(section):
    return f'{int(section):02d}' if str(section).isdigit() else str(section)

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else (o.tolist() if hasattr(o, 'tolist') else str(o))

def _same(a, b, tol=1e-3):
    """Equal up to a relative tolerance of 0.1% for numbers (results are rounded differently in places)."""
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
        return abs(a - b) <= tol * max(1.0, abs(b))
    return a == b

def _short(v, n=160):
    s = json.dumps(v, default=_jsonable)
    return s if len(s) <= n else s[:n] + ' ...'

def log_result(section, key, value, note=''):
    """Store one reportable number under results[section][key]; earlier values are kept as history.
    REBUILD = False: a key that is already saved is never overwritten. The new value is compared with it
    instead, so a rerun shows that it reproduces the saved number. DEBUG runs write to results_debug.json."""
    path = Path(CFG['paths']['results'])
    if CFG['DEBUG']:
        path = path.with_name('results_debug.json')
    res = json.loads(path.read_text()) if path.exists() else {}
    sec = res.setdefault(_sec(section), {})
    new = json.loads(json.dumps(value, default=_jsonable))
    if key in sec and not rebuild():
        old = sec[key]
        if _same(new, old['value']):
            print(f"✓ [{_sec(section)}] {key} = {_short(new)}  (matches the saved result of {old['time'][:10]})")
        else:
            print(f"≠ [{_sec(section)}] {key}: now {_short(new)} | saved {_short(old['value'])} "
                  f"({old['time'][:10]}; saved value kept, REBUILD = True overwrites)")
        return value
    entry = {'value': new, 'time': datetime.now().isoformat(timespec='seconds'), 'note': note}
    if key in sec:
        entry['history'] = sec[key].get('history', []) + [{'value': sec[key]['value'], 'time': sec[key]['time']}]
    sec[key] = entry
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(res, indent=2, default=_jsonable))
    os.replace(tmp, path)
    print(f"logged [{_sec(section)}] {key} = {_short(new)}" + ('  (DEBUG file)' if CFG['DEBUG'] else ''))
    return value

def saved_result(section, key):
    """The saved value of results[section][key] (None if missing)."""
    res = json.loads(Path(CFG['paths']['results']).read_text())
    return res.get(_sec(section), {}).get(key, {}).get('value')

def save_fig(fig, section, name, show=True):
    """Save a figure as figures/<section>_<name>.png at dpi 200 (figures/debug/ when DEBUG)."""
    import matplotlib.pyplot as plt
    folder = Path(CFG['paths']['figures']) / ('debug' if CFG['DEBUG'] else '')
    folder.mkdir(parents=True, exist_ok=True)
    out = folder / f'{_sec(section)}_{name}.png'
    fig.savefig(out, dpi=200, bbox_inches='tight')
    print('saved figure:', out)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return out

def show_saved(section, figs=(), keys=()):
    """Show figures and logged numbers that a step saved when it was built (used when the step is not rerun)."""
    from IPython.display import display, Image as IPImage
    for k in keys:
        res = json.loads(Path(CFG['paths']['results']).read_text()).get(_sec(section), {}).get(k)
        print(f"saved [{_sec(section)}] {k} = {_short(res['value'], 400)}  ({res['time'][:10]})" if res
              else f'not logged yet: [{_sec(section)}] {k}')
    for n in figs:
        p = Path(CFG['paths']['figures']) / f'{_sec(section)}_{n}.png'
        if p.exists():
            made = datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d')
            print(f'saved figure {p.name} (made {made}):'); display(IPImage(str(p), width=1000))
        else:
            print('figure not found:', p.name)

def done(*paths):
    """True if every output already exists, so a resumable cell can skip finished work."""
    ok = all(Path(p).exists() for p in paths)
    if ok:
        print('skip (already done):', ', '.join(Path(p).name for p in paths))
    return ok

def build_needed(*paths):
    """True = run this step: REBUILD is True, or one of its outputs is missing.
    False = the outputs are saved and REBUILD is False, so the cell loads them instead of making them again."""
    missing = [Path(p).name for p in paths if not Path(p).exists()]
    if rebuild() or missing:
        if missing and not rebuild():
            print('not saved yet, building:', ', '.join(missing))
        return True
    print('loading saved outputs (REBUILD = False):', ', '.join(Path(p).name for p in paths))
    return False

def may_write(path):
    """True if this cell may write `path`: REBUILD is True or the file does not exist yet.
    With REBUILD = False a saved file is never overwritten, so a rerun cannot change data the models used."""
    if rebuild() or not Path(path).exists():
        return True
    print('kept saved file (REBUILD = False):', Path(path).name)
    return False

def run_info(folder):
    """One line about a saved training run: folder name and the date its files were last written."""
    folder = Path(folder)
    files = [p for p in folder.rglob('*') if p.is_file()]
    if not files:
        return f'{folder.name}: no saved files'
    t = max(p.stat().st_mtime for p in files)
    return f"{folder.name}: saved run, last written {datetime.fromtimestamp(t):%Y-%m-%d %H:%M}"

def debug_subset(df, by, n=None):
    """In DEBUG mode keep the first n rows per group (e.g. per class or per video); otherwise return df."""
    if not CFG['DEBUG']:
        return df
    return df.groupby(by, group_keys=False).head(n or CFG['debug_n']).reset_index(drop=True)

Overwriting /content/drive/MyDrive/AAI3001_Group9/repo/notebooks/nbtools.py


In [6]:
%%writefile /content/drive/MyDrive/AAI3001_Group9/repo/notebooks/restore.py

# Cell 1.6 · restore.py

"""Body of the restore cell at the top of every section (run with exec, so every name lands in the notebook).
Needs ROOT and REBUILD to be set first. Installs what is missing, imports, reloads CFG and the helpers."""
import os, sys, json, time, math, random, copy, shutil, zipfile, glob, re, subprocess, importlib, warnings
import importlib.metadata as md
from pathlib import Path
from datetime import datetime
ROOT = Path(ROOT)

def ver(dist):
    try:
        return md.version(dist)
    except md.PackageNotFoundError:
        return None

_need = [d for d in ['ultralytics', 'segmentation-models-pytorch', 'onnx', 'onnxruntime', 'grad-cam', 'kagglehub']
         if ver(d) is None]
from packaging.version import Version
if ver('transformers') is None or Version(ver('transformers')) < Version('4.56'):
    _need.append('transformers>=4.56')
if _need:
    print('installing:', _need)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_need], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import torchvision, torchvision.transforms as transforms
from torchvision import models
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from tqdm.auto import tqdm
warnings.filterwarnings('ignore', category=UserWarning)

for _p in [ROOT / 'repo' / 'notebooks', ROOT / 'repo']:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
if str(ROOT / 'repo' / 'src') not in sys.path:
    sys.path.append(str(ROOT / 'repo' / 'src'))
importlib.invalidate_caches()
import nbtools; importlib.reload(nbtools)
from nbtools import (seed_everything, log_result, save_fig, done, debug_subset, save_cfg, load_cfg,
                     build_needed, may_write, show_saved, saved_result, run_info)

CFG = load_cfg(ROOT / 'repo' / 'configs' / 'cfg.json')
_old = CFG.get('root', '/content/drive/MyDrive/AAI3001_Group9')      # re-point saved paths if ROOT moved
CFG['paths'] = {k: v.replace(_old, str(ROOT), 1) for k, v in CFG['paths'].items()}
CFG['root'], CFG['DEBUG'], CFG['REBUILD'] = str(ROOT), False, bool(REBUILD)
nbtools.init(CFG)
seed_everything(CFG['seed'])
P, PRODUCE, TIERS = CFG['paths'], CFG['produce_classes'], CFG['tiers']
LAB = Path(P['labels'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"restored | ROOT={ROOT} | REBUILD={CFG['REBUILD']} | device={device} | "
      f"transformers {ver('transformers')} | {len(PRODUCE)} types")

Writing /content/drive/MyDrive/AAI3001_Group9/repo/notebooks/restore.py


In [7]:
# Cell 1.7 · Load the helpers, seed everything; save CFG only if it is new (or REBUILD = True)

for p in [str(ROOT / 'repo' / 'notebooks'), str(ROOT / 'repo')]:
    if p not in sys.path:
        sys.path.insert(0, p)
import importlib, nbtools
importlib.reload(nbtools)
from nbtools import seed_everything, log_result, save_fig, done, debug_subset, save_cfg, load_cfg, may_write

def same(base, saved):
    """True if every key of `base` has the same value in `saved` (saved may hold extra keys)."""
    if isinstance(base, dict):
        return isinstance(saved, dict) and all(k in saved and same(v, saved[k]) for k, v in base.items())
    return base == saved

CFG['REBUILD'] = REBUILD
nbtools.init(CFG)
seed_everything(CFG['seed'])
CFG_FILE = ROOT / 'repo' / 'configs' / 'cfg.json'
if may_write(CFG_FILE):
    save_cfg(CFG, CFG_FILE)
    print('CFG saved to repo/configs/cfg.json')
else:
    saved = load_cfg(CFG_FILE)
    diff = [k for k in CFG if k not in ('DEBUG', 'REBUILD', 'root') and not same(CFG[k], saved.get(k))]
    print('cfg.json agrees with cell 1.3' if not diff else f'⚠️ cfg.json differs from cell 1.3 in: {diff}')
    CFG = {**saved, 'DEBUG': CFG['DEBUG'], 'REBUILD': REBUILD, 'root': str(ROOT)}   # saved file also holds the frozen split
    nbtools.init(CFG)
P, PRODUCE, TIERS = CFG['paths'], CFG['produce_classes'], CFG['tiers']
print('helpers ready | seed', CFG['seed'], '| results file:', 'results_debug.json' if CFG['DEBUG'] else 'results.json')

kept saved file (REBUILD = False): cfg.json
cfg.json agrees with cell 1.3
helpers ready | seed 42 | results file: results.json


In [8]:
%%writefile /content/drive/MyDrive/AAI3001_Group9/repo/src/preprocess.py

# Cell 1.8 · preprocess.py

"""Image preprocessing shared by the notebook and the app, so both feed the models identical numbers.
Matches torchvision: Resize(255) -> CenterCrop(224) -> ToTensor -> Normalize(ImageNet mean/std)."""
import numpy as np
from PIL import Image

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

def to_rgb(img):
    """PIL image or HxWx3 array -> RGB PIL image; transparent pixels become white (as in the hurdle)."""
    if isinstance(img, np.ndarray):
        img = Image.fromarray(img)
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        img = img.convert('RGBA')
        bg = Image.new('RGB', img.size, (255, 255, 255))
        bg.paste(img, mask=img.getchannel('A'))
        return bg
    return img.convert('RGB')

def resize_center_crop(img, size=224, resize=255):
    """Shorter side -> resize (bilinear), then a centre square of size x size."""
    w, h = img.size
    nw, nh = (resize, int(resize * h / w)) if w <= h else (int(resize * w / h), resize)
    img = img.resize((nw, nh), Image.BILINEAR)
    left, top = int(round((nw - size) / 2.0)), int(round((nh - size) / 2.0))
    return img.crop((left, top, left + size, top + size))

def normalize(img, mean=MEAN, std=STD):
    """RGB PIL -> float32 CHW array, scaled to [0, 1] then standardised (L8 slide 13)."""
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = (x - np.asarray(mean, dtype=np.float32)) / np.asarray(std, dtype=np.float32)
    return x.transpose(2, 0, 1)

def preprocess(img, size=224, resize=255):
    """Photo or crop -> float32 array of shape (1, 3, size, size), ready for an ONNX model."""
    return normalize(resize_center_crop(to_rgb(img), size, resize))[None]

def crop_box(img, box, pad=0.10):
    """Cut one detected item (x1, y1, x2, y2 in pixels) with a small margin, clipped to the image."""
    img = to_rgb(img)
    W, H = img.size
    x1, y1, x2, y2 = box
    pw, ph = (x2 - x1) * pad, (y2 - y1) * pad
    return img.crop((max(0, int(x1 - pw)), max(0, int(y1 - ph)), min(W, int(x2 + pw)), min(H, int(y2 + ph))))

Overwriting /content/drive/MyDrive/AAI3001_Group9/repo/src/preprocess.py


In [9]:
# Cell 1.9 · Prove preprocess.py == the training transform; smoke-test log_result and save_fig

import src.preprocess as pp
importlib.reload(pp)

I = CFG['img']
test_tf = transforms.Compose([transforms.Resize(I['resize']), transforms.CenterCrop(I['size']),
                              transforms.ToTensor(), transforms.Normalize(I['mean'], I['std'])])
rng = np.random.default_rng(CFG['seed'])
diffs = []
for w, h in [(640, 480), (480, 640), (300, 300), (1000, 333), (226, 900)]:
    img = Image.fromarray(rng.integers(0, 256, (h, w, 3), dtype=np.uint8))
    app_x = pp.preprocess(img, I['size'], I['resize'])
    train_x = test_tf(img).unsqueeze(0).numpy()
    diffs.append(float(np.abs(app_x - train_x).max()))
    print(f'{w:>4}x{h:<4} shapes {app_x.shape} vs {train_x.shape} | max |app - torchvision| = {diffs[-1]:.2e}')
assert max(diffs) < 1e-5, 'preprocess.py does not match the training transform'

rgba = Image.new('RGBA', (50, 40), (255, 0, 0, 0))                  # fully transparent -> white
assert np.asarray(pp.to_rgb(rgba))[0, 0].tolist() == [255, 255, 255]
print('✅ preprocess.py matches the training transform; transparency -> white')

fig, ax = plt.subplots(figsize=(3, 2))
ax.plot([0, 1], [0, 1]); ax.set_title('save_fig smoke test')
save_fig(fig, 2, 'smoke_test', show=False)
log_result(2, 'preprocess_parity_max_abs_diff', max(diffs))
log_result(2, 'environment', {'gpu': gpu, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
                              'ultralytics': ver('ultralytics'), 'transformers': ver('transformers'),
                              'onnxruntime': ver('onnxruntime'), 'python': sys.version.split()[0]})

 640x480  shapes (1, 3, 224, 224) vs (1, 3, 224, 224) | max |app - torchvision| = 0.00e+00
 480x640  shapes (1, 3, 224, 224) vs (1, 3, 224, 224) | max |app - torchvision| = 0.00e+00
 300x300  shapes (1, 3, 224, 224) vs (1, 3, 224, 224) | max |app - torchvision| = 0.00e+00
1000x333  shapes (1, 3, 224, 224) vs (1, 3, 224, 224) | max |app - torchvision| = 0.00e+00
 226x900  shapes (1, 3, 224, 224) vs (1, 3, 224, 224) | max |app - torchvision| = 0.00e+00
✅ preprocess.py matches the training transform; transparency -> white
saved figure: /content/drive/MyDrive/AAI3001_Group9/figures/02_smoke_test.png
logged [02] preprocess_parity_max_abs_diff = 0.0
logged [02] environment = {"gpu": "none", "torch": "2.14.1+cpu", "torchvision": "0.29.1+cpu", "ultralytics": "8.4.216", "transformers": "5.22.1", "onnxruntime": "1.32.0", "python": "3.13 ...


{'gpu': 'none',
 'torch': '2.14.1+cpu',
 'torchvision': '0.29.1+cpu',
 'ultralytics': '8.4.216',
 'transformers': '5.22.1',
 'onnxruntime': '1.32.0',
 'python': '3.13.18'}

# **Dataset Preparation**

The hurdle used one clean Kaggle dataset with two labels. The scale needs more: items in context, a freshness scale between fresh and rotten, boxes and blemish masks. This section builds all of it from three sources and keeps every test set apart before any model is trained.

# **Data Augmentation**

# **Goal 1: Produce Detection**

## **Baseline Model**

## **Fine-tuning**

## **Evaluation**

## **Ablations**

## **Results/Interpretations**

# **Goal 2: Freshness Percentage**

## **Baseline Model**

## **Fine-tuning**

## **Evaluation**

## **Ablations**

## **Results/Interpretations**

# **Goal 3: Blemish Segmentation**

## **Baseline Model**

## **Fine-tuning**

## **Evaluation**

## **Ablations**

## **Results/Interpretations**

# **Goal 4: Image Generator**

## **Baseline Model**

## **Fine-tuning**

## **Evaluation**

## **Ablations**

## **Results/Interpretations**

# **Model Deployment**